# Log-Target + Remove Near-Constant Features

Train a simple `LGBMRegressor` on `log1p(target)` after removing features where the share of zeros is greater than `99%`.

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [2]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
ZERO_SHARE_THRESHOLD = 0.99

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [6]:
X_train_full, X_test_full, y_train, y_test, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

zero_share_train = (X_train_full == 0).mean()
selected_features = zero_share_train[zero_share_train <= ZERO_SHARE_THRESHOLD].index
removed_features = zero_share_train[zero_share_train > ZERO_SHARE_THRESHOLD].index

X_train = X_train_full[selected_features]
X_test = X_test_full[selected_features]

pd.DataFrame(
    {
        "metric": ["features_before", "features_removed", "features_after"],
        "value": [X.shape[1], len(removed_features), X_train.shape[1]],
    }
)

,metric,value
0,features_before,4731
1,features_removed,2062
2,features_after,2669


In [7]:
cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [8]:
model = LGBMRegressor(
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

In [9]:
# CV RMSE in log-space is equivalent to RMSLE for this setup.
cv_scores_rmsle = -cross_val_score(
    estimator=model,
    X=X_train,
    y=y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

pd.DataFrame(
    {
        "metric": ["cv_rmsle_mean", "cv_rmsle_std"],
        "value": [cv_scores_rmsle.mean(), cv_scores_rmsle.std()],
    }
)

,metric,value
0,cv_rmsle_mean,1.469250
1,cv_rmsle_std,0.034585


In [10]:
model.fit(X_train, y_train_log)

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [11]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test, y_pred),
            root_mean_squared_error(y_test, y_pred),
            mean_absolute_error(y_test, y_pred),
            r2_score(y_test, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.3f}"})

,metric,value
0,rmsle,1.483
1,rmse,"7,337,001.010"
2,mae,"4,173,037.371"
3,r2,0.157


## Conclusion

- Near-constant feature filtering is now fit on the training split only, then applied to the holdout split, so the evaluation no longer leaks information from test into feature selection.
- The combined setup removes `2,104` near-constant sparse features and reduces the feature space from `4,731` to `2,627` columns.
- Since the task metric is `RMSLE`, the key result is `CV RMSLE = 1.465 +/- 0.030` and `test RMSLE = 1.483`.
- After inverse transformation back to raw target space, the model gives `RMSE = 7,295,799.329`, `MAE = 4,185,374.138`, and `R2 = 0.166`.
- Compared with the raw-target branches, this combined approach is much stronger on the competition metric.
- Compared with the current baseline in `03_baseline.ipynb` (`CV RMSLE = 1.472 +/- 0.035`, `test RMSLE = 1.482`), removing near-constant features gives a small cross-validation improvement but a slightly worse held-out test `RMSLE`.
- Conclusion: combining `log1p(target)` with removal of features that have more than `99%` zeros is a valid and competitive experiment, but the current evidence still favors the simpler log-target baseline from `03_baseline.ipynb` as the reference setup.